In [3]:
import duckdb
import polars as pl
import pandas as pd

In [ ]:
con = duckdb.connect()

con.sql("""
    COPY (
        WITH lineas AS (
            SELECT 
                line,
                -- Separa los campos restantes a partir del carácter 30 por espacios
                string_split_regex(trim(substr(line, 30)), '\s+') AS cols
            FROM read_csv('C:/Users/rafah/Documents/Deudores BCRA/deudores.txt', header=False, columns={'line': 'VARCHAR'})
        )
        SELECT 
            -- 1. Encabezado de Ancho Fijo (Primeros 29 caracteres)
            substr(line, 1, 5)                       AS entidad,
            substr(line, 6, 6)                       AS periodo,
            substr(line, 12, 2)                      AS tipo_doc,
            substr(line, 14, 11)                     AS cuit,
            TRY_CAST(substr(line, 25, 3) AS INT)     AS actividad,
            TRY_CAST(substr(line, 28, 2) AS INT)     AS situacion,

            -- 2. Montos e Indicadores (Campos 7 al 24)
            TRY_CAST(REPLACE(cols[1], ',', '.') AS DOUBLE)  AS prestamos_total_garantias,
            TRY_CAST(REPLACE(cols[2], ',', '.') AS DOUBLE)  AS sin_uso,
            TRY_CAST(REPLACE(cols[3], ',', '.') AS DOUBLE)  AS garantias_otorgadas,
            TRY_CAST(REPLACE(cols[4], ',', '.') AS DOUBLE)  AS otros_conceptos,
            TRY_CAST(REPLACE(cols[5], ',', '.') AS DOUBLE)  AS garantias_preferidas_a,
            TRY_CAST(REPLACE(cols[6], ',', '.') AS DOUBLE)  AS garantias_preferidas_b,
            TRY_CAST(REPLACE(cols[7], ',', '.') AS DOUBLE)  AS sin_garantias_preferidas,
            TRY_CAST(REPLACE(cols[8], ',', '.') AS DOUBLE)  AS contragarantias_preferidas_a,
            TRY_CAST(REPLACE(cols[9], ',', '.') AS DOUBLE)  AS contragarantias_preferidas_b,
            TRY_CAST(REPLACE(cols[10], ',', '.') AS DOUBLE) AS sin_contragarantias_preferidas,
            TRY_CAST(REPLACE(cols[11], ',', '.') AS DOUBLE) AS previsiones,
            TRY_CAST(cols[12] AS INT)                       AS deuda_cubierta,
            TRY_CAST(cols[13] AS INT)                       AS proceso_judicial_revision,
            TRY_CAST(cols[14] AS INT)                       AS refinanciaciones,
            TRY_CAST(cols[15] AS INT)                       AS recategorizacion_obligatoria,
            TRY_CAST(cols[16] AS INT)                       AS situacion_juridica,
            TRY_CAST(cols[17] AS INT)                       AS irrecuperables_tecnica,
            TRY_CAST(cols[18] AS INT)                       AS dias_atraso
        FROM lineas
    ) TO 'C:/Users/rafah/Documents/Deudores BCRA/deudores_clean.parquet' (FORMAT PARQUET)
""")

con.close()

<positron-console-cell-2>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


In [ ]:

con = duckdb.connect()

con.sql("""
    COPY (
        WITH lineas AS (
            SELECT line
            FROM read_csv(
                'C:/Users/rafah/Documents/Deudores BCRA/Padron_ARCA.txt', 
                header=False, 
                columns={'line': 'VARCHAR'},
                quote='',
                escape='',
                encoding='CP1252',
                ignore_errors=true,
                auto_detect=False
            )
        )
        SELECT 
            -- 1. Identificación
            trim(substr(line, 1, 11))                           AS cuit,
            trim(substr(line, 12, 160))                         AS denominacion,
            TRY_CAST(trim(substr(line, 172, 6)) AS INT)         AS actividad,
            
            -- 2. Estado y Reemplazo
            NULLIF(trim(substr(line, 178, 1)), '')              AS marca_baja,
            NULLIF(trim(substr(line, 179, 11)), '')             AS cuit_reemplazo,
            
            -- 3. Fechas y Demografía
            TRY_CAST(
                NULLIF(trim(substr(line, 190, 10)), '1901-01-01') 
                AS DATE
            )                                                   AS fecha_nac_contrato,
            
            NULLIF(trim(substr(line, 200, 1)), '')              AS sexo,
            NULLIF(trim(substr(line, 201, 10)), '')             AS codigo_postal,
            TRY_CAST(trim(substr(line, 211, 2)) AS INT)         AS provincia,
            
            TRY_STRPTIME(
                NULLIF(trim(substr(line, 213, 8)), '19010101'), 
                '%Y%m%d'
            )::DATE                                             AS fecha_fallecimiento

        FROM lineas
    ) TO 'C:/Users/rafah/Documents/Deudores BCRA/padron_arca_clean.parquet' (FORMAT PARQUET, COMPRESSION 'ZSTD')
""")

con.close()

IOException: IO Error: No files found that match the pattern "Padron_ARCA.txt"

LINE 5:             FROM read_csv(
                         ^

In [13]:
schema = duckdb.sql("DESCRIBE SELECT * FROM 'deudores_clean.parquet'").df()
print(schema)

                       column_name column_type null   key default extra
0                          entidad     VARCHAR  YES  None    None  None
1                          periodo     VARCHAR  YES  None    None  None
2                         tipo_doc     VARCHAR  YES  None    None  None
3                             cuit     VARCHAR  YES  None    None  None
4                        actividad     INTEGER  YES  None    None  None
5                        situacion     INTEGER  YES  None    None  None
6        prestamos_total_garantias      DOUBLE  YES  None    None  None
7                          sin_uso      DOUBLE  YES  None    None  None
8              garantias_otorgadas      DOUBLE  YES  None    None  None
9                  otros_conceptos      DOUBLE  YES  None    None  None
10          garantias_preferidas_a      DOUBLE  YES  None    None  None
11          garantias_preferidas_b      DOUBLE  YES  None    None  None
12        sin_garantias_preferidas      DOUBLE  YES  None    Non

In [14]:
schema_padron = duckdb.sql("DESCRIBE SELECT * FROM 'padron_arca_clean.parquet'").df()
print(schema_padron)

           column_name column_type null   key default extra
0                 cuit     VARCHAR  YES  None    None  None
1         denominacion     VARCHAR  YES  None    None  None
2            actividad     INTEGER  YES  None    None  None
3           marca_baja     VARCHAR  YES  None    None  None
4       cuit_reemplazo     VARCHAR  YES  None    None  None
5   fecha_nac_contrato        DATE  YES  None    None  None
6                 sexo     VARCHAR  YES  None    None  None
7        codigo_postal     VARCHAR  YES  None    None  None
8            provincia     INTEGER  YES  None    None  None
9  fecha_fallecimiento        DATE  YES  None    None  None


In [ ]:
con = duckdb.connect()

con.sql("""
    COPY (
        WITH base AS (
            SELECT 
                d.periodo,
                d.entidad,
                d.situacion,
                p.provincia,
                p.sexo,
                d.cuit,
                CASE 
                    WHEN CAST(d.cuit AS VARCHAR) LIKE '3%' THEN 'Jurídica'
                    WHEN CAST(d.cuit AS VARCHAR) LIKE '2%' THEN 'Física'
                    ELSE 'Otra/Desconocida'
                END AS tipo_persona,
                CASE 
                    WHEN CAST(d.cuit AS VARCHAR) LIKE '3%' THEN CAST(p.actividad AS VARCHAR)
                    ELSE 'N/A' 
                END AS actividad,
                d.prestamos_total_garantias
            FROM 
                'C:/Users/SYC/Downloads/deudores/deudores_clean.parquet' d
            LEFT JOIN 
                'C:/Users/SYC/Downloads/deudores/padron_arca_clean.parquet' p 
                ON d.cuit = p.cuit
        )
        SELECT 
            entidad,
            periodo,
            situacion,
            provincia,
            sexo,
            tipo_persona,
            actividad,
            SUM(prestamos_total_garantias) AS prestamos,
            COUNT(DISTINCT cuit) AS cantidad_deudores
        FROM base
        GROUP BY 
            entidad,
            periodo,
            situacion,
            provincia,
            sexo,
            tipo_persona,
            actividad
    ) TO 'data/deuda_new.parquet' (FORMAT PARQUET);
""")

con.close()

In [ ]:
con = duckdb.connect()

con.sql("""
    COPY (
        WITH base AS (
            SELECT 
                d.periodo,
                d.entidad,
                d.situacion,
                p.provincia,
                p.sexo,
                d.cuit,
                CASE 
                    WHEN CAST(d.cuit AS VARCHAR) LIKE '3%' THEN 'Jurídica'
                    WHEN CAST(d.cuit AS VARCHAR) LIKE '2%' THEN 'Física'
                    ELSE 'Otra/Desconocida'
                END AS tipo_persona,
                CASE 
                    WHEN CAST(d.cuit AS VARCHAR) LIKE '3%' THEN CAST(p.actividad AS VARCHAR)
                    ELSE 'N/A' 
                END AS actividad,
                -- Cálculo de rango etario solo para Personas Físicas con fecha válida
                CASE 
                    WHEN CAST(d.cuit AS VARCHAR) NOT LIKE '2%' THEN 'N/A (Jurídica)'
                    WHEN p.fecha_nac_contrato IS NULL THEN 'Sin dato'
                    ELSE 
                        CASE 
                            -- Calcula la edad a partir de la fecha de nacimiento
                            WHEN date_diff('year', TRY_CAST(p.fecha_nac_contrato AS DATE), CURRENT_DATE) < 25 THEN '<25'
                            WHEN date_diff('year', TRY_CAST(p.fecha_nac_contrato AS DATE), CURRENT_DATE) BETWEEN 25 AND 34 THEN '25-34'
                            WHEN date_diff('year', TRY_CAST(p.fecha_nac_contrato AS DATE), CURRENT_DATE) BETWEEN 35 AND 44 THEN '35-44'
                            WHEN date_diff('year', TRY_CAST(p.fecha_nac_contrato AS DATE), CURRENT_DATE) BETWEEN 45 AND 54 THEN '45-54'
                            WHEN date_diff('year', TRY_CAST(p.fecha_nac_contrato AS DATE), CURRENT_DATE) BETWEEN 55 AND 64 THEN '55-64'
                            WHEN date_diff('year', TRY_CAST(p.fecha_nac_contrato AS DATE), CURRENT_DATE) >= 65 THEN '>65'
                            ELSE 'Sin dato'
                        END
                END AS rango_etario,
                d.prestamos_total_garantias,
                CASE 
                    WHEN d.prestamos_total_garantias <= 10 THEN '0-10'
                    WHEN d.prestamos_total_garantias <= 50 THEN '10-50'
                    WHEN d.prestamos_total_garantias <= 100 THEN '50-100'
                    WHEN d.prestamos_total_garantias <= 500 THEN '100-500'
                    WHEN d.prestamos_total_garantias <= 1000 THEN '500-1000'
                    WHEN d.prestamos_total_garantias <= 5000 THEN '1000-5000'
                    WHEN d.prestamos_total_garantias <= 10000 THEN '5000-10000'
                    ELSE '>10000'
                END AS rango_prestamo
            FROM 
                'C:/Users/SYC/Downloads/deudores/deudores_clean.parquet' d
            LEFT JOIN 
                'C:/Users/SYC/Downloads/deudores/padron_arca_clean.parquet' p 
                ON d.cuit = p.cuit
        )
        SELECT 
            entidad,
            periodo,
            situacion,
            provincia,
            sexo,
            tipo_persona,
            actividad,
            rango_etario,
            SUM(prestamos_total_garantias) AS prestamos,
            MEDIAN(prestamos_total_garantias) AS mediana_prestamos,
            COUNT(DISTINCT cuit) AS cantidad_deudores
        FROM base
        GROUP BY 
            entidad,
            periodo,
            situacion,
            provincia,
            sexo,
            tipo_persona,
            actividad,
            rango_etario,
            rango_prestamo
    ) TO 'data/deuda_new.parquet' (FORMAT PARQUET);
""")

con.close()

In [ ]:
padron = pl.scan_parquet("padron_arca_clean.parquet")
deudores = pl.scan_parquet("deudores_clean.parquet")

padron_filtrado = padron.filter(pl.col("provincia") == 3)

cruce_lazy = padron_filtrado.join(deudores,on="cuit",how="left")

#cruce_lazy.sink_parquet("cordoba.parquet")